# MUSANGA V3.1 · NB05 — Baego full-year Planet NICFI pilot

**Current run mode: Baego 2023 only**. Existing Q1/Q2/Q3/Q4 exports are retained in place. New Sections **07–10** catalogue all 12 months; export only January, February, April, May, July, August, October and November; construct a cloud-screened annual median; and compute per-month spatial texture **before** annual aggregation. No other forest is exported until this Baego experiment is assessed.

**Data layout** (all paths beneath `Final/Outputs/V3_1/05_sat_predictor_exploration/`):

- `input_layers/Musanga_V3_1_05_Planet_NICFI_Monthly/`: twelve original 4-band monthly raster products (four already present).
- `input_layers/Musanga_V3_1_05_Planet_Annual/`: annual reflectance median plus observation counts and quality-screen metrics.
- `derived_features/Musanga_V3_1_05_Planet_Texture/`: annual median of *per-month* texture features, plus their temporal variability.
- `07_baego_full_year_catalogue.csv` and dedicated export registry: provenance and restartability.

**Cloud-screen limitation:** NICFI has no matching UDM2 mask in the accessed four-band GEE asset. The annual pipeline screens **relative temporal brightness anomalies**, not guaranteed cloud labels. It does not promise a perfectly cloud-free product. Annual texture uses masked neighbourhood statistics. Compare screening on/off and verify image detail and valid counts visually before modelling.

**Pilot:** 4.77 m native nominal scale, ROI.shp intersected with UAV mosaic extent plus 250 m, and Earth Engine project `musanga-511110`. No change to NB04/04B training or input data. 


## 00 · Settings: explicit site-year and quarter-month decisions

In [ ]:
# Edit only these settings for normal use.
EE_PROJECT = 'musanga-511110'  # Same Earth Engine Cloud project as V3.1 NB04/04A.
YEAR_BY_SITE = {'Baego': 2023, 'MOS': 2024, 'Yangambi': 2024, 'Yoko': 2020}
QUARTER_MONTHS = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
BUFFER_M = 250.0
EXPORT_CRS = 'EPSG:32635'  # Forest sites around Kisangani; metric output raster.
EXPORT_SCALE_M = 4.77
FILE_DIMENSIONS = 4096    # Creates multiple GeoTIFF pieces for large AOIs.
DRIVE_EXPORT_FOLDER = 'Musanga_V3_1_05_Planet_NICFI_Monthly'  # Unique folder name; precreated inside Final/Outputs/V3_1/05_sat_predictor_exploration/input_layers/.
RUN_SITES = ['Baego']  # PILOT: only Baego; other-site exports disabled until pilot passes.
SUBMIT_EXPORTS = False    # Legacy quarterly export only; existing Baego quarterly rasters are preserved.
SUBMIT_MISSING_BAEGO_MONTHS = False  # Turn True after reviewing new Section 07 catalogue.
SUBMIT_ANNUAL_PRODUCTS = False  # Turn True after reviewing annual composition plan.
MAX_NEW_TASKS_PER_RUN = 4 # Avoid queuing a huge set of full-resolution exports at once.
SKIP_EXISTING_OR_ACTIVE = True
SOURCE_COLLECTION = 'projects/planet-nicfi/assets/basemaps/africa'
BANDS = ['B', 'G', 'R', 'N']
print('Quarter months:', QUARTER_MONTHS, ' | submitting:', SUBMIT_EXPORTS)

## 01 · Colab environment, Drive and Earth Engine

**Important:** This notebook does not run `pip` or change `google-auth`. If you previously ran the old cell, choose **Runtime → Disconnect and delete runtime**, reconnect, and then **Runtime → Run all**. You will be asked to authorize Earth Engine and, unless entered in Settings, provide your Google Cloud **project ID**.

In [ ]:
# Colab includes these packages. Do NOT pip-install or upgrade google-auth / earthengine-api here.
# If packages were upgraded earlier, use Runtime > Disconnect and delete runtime, then reconnect.
from google.colab import drive
import ee, geopandas as gpd, rasterio, pandas as pd
import json, hashlib, os, calendar, datetime, re
from pathlib import Path
from shapely.geometry import box, mapping
from shapely.ops import unary_union
from IPython.display import display

drive.mount('/content/drive')
ROOT = next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                             Path('/content/drive/MyDrive/Musanga/Final')]
             if p.is_dir()), None)
assert ROOT is not None, 'Musanga/Final not found in Google Drive.'
BASE = ROOT/'Outputs/V3_1'
MANIFEST_PATH = BASE/'00_input_audit/00_input_manifest.json'
assert MANIFEST_PATH.is_file(), 'Run NB00 audit, or correct MANIFEST_PATH.'
CACHE_META = BASE/'05_sat_predictor_exploration'
CACHE_META.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = CACHE_META/'input_layers'
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DRIVE = INPUT_ROOT/DRIVE_EXPORT_FOLDER
CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
# Earth Engine toDrive(folder=...) resolves by FOLDER NAME, not by full path.
# The unique precreated folder under the Musanga workspace avoids ambiguous names.
# Google Drive folder indexing may be delayed: check actual target on first export.
# Explicit user authentication, followed by initialization using a Cloud project ID.
# Earth Engine now requires the project even when login succeeds.
if not EE_PROJECT or EE_PROJECT == 'your-ee-project-id':
    EE_PROJECT = input('Enter your Earth Engine Google Cloud PROJECT ID: ').strip()
if not EE_PROJECT:
    raise ValueError('Project ID is required. Find it in the Earth Engine Code Editor project selector.')
print('Authenticating Earth Engine for project:', EE_PROJECT)
ee.Authenticate(auth_mode='colab')
ee.Initialize(project=EE_PROJECT)
COLLECTION = ee.ImageCollection(SOURCE_COLLECTION)
print('ROOT:', ROOT)
print('Source manifest:', MANIFEST_PATH)
print('Cache metadata:', CACHE_META)
print('Planet raw input layers:', CACHE_DRIVE)
print('Planet collection size (permission test):', COLLECTION.size().getInfo())

## 02 · Site-specific ROI × orthomosaic footprints + 250 m buffer

In [ ]:
A = json.loads(MANIFEST_PATH.read_text())
ROI_PATH = Path(A['roi'])
assert ROI_PATH.exists(), f'Missing authoritative ROI: {ROI_PATH}'
ROI = gpd.read_file(ROI_PATH)
assert ROI.crs is not None and len(ROI) > 0
ROI = ROI[ROI.geometry.notna() & ~ROI.geometry.is_empty].copy()
# Scope to the same 5 source RGB mosaics used by V3.1 NB02B.
MOSAIC_ALLOW = {
 'Baego.tif': 'Baego',
 'Mos_1_2_transparent_mosaic_group1_georef.tif': 'MOS',
 'Mos_5_6_transparent_mosaic_group1_georef.tif': 'MOS',
 'Yangambi_georef.tif': 'Yangambi',
 'Yoko_kvo_georef.tif': 'Yoko',
}
records = []
intersections = {site:[] for site in YEAR_BY_SITE}
for row in A['mosaics']:
    name = row['file']
    if name not in MOSAIC_ALLOW:
        continue
    path = Path(row['path'])
    assert path.exists(), f'Missing RGB mosaic: {path}'
    site = MOSAIC_ALLOW[name]
    with rasterio.open(path) as ds:
        assert ds.crs, f'No raster CRS: {name}'
        # Raster's rectangular geo-referenced *bounding* footprint, not the valid-pixel mask.
        raster_envelope = box(*ds.bounds)
        r = ROI.to_crs(ds.crs)
        roi_union = unary_union([g for g in r.geometry if g is not None and not g.is_empty])
        clipped = roi_union.intersection(raster_envelope)
        records.append({'site':site, 'mosaic':name, 'raster_crs':str(ds.crs),
                        'roi_intersection_m2_in_raster_crs':float(clipped.area),
                        'raster_width':ds.width,'raster_height':ds.height})
        if not clipped.is_empty:
            # Store pieces in a shared geographic CRS until per-site metric buffering.
            gg = gpd.GeoSeries([clipped], crs=ds.crs).to_crs('EPSG:4326').iloc[0]
            intersections[site].append(gg)

display(pd.DataFrame(records))
AOI = {}
audit = []
for site in RUN_SITES:
    assert intersections[site], f'No ROI/footprint intersection for {site}; inspect ROI.shp and input registry.'
    united_wgs84 = unary_union(intersections[site])
    base = gpd.GeoSeries([united_wgs84], crs='EPSG:4326')
    # For these DRC forests projected UTM 35N is appropriate; estimate per-site to avoid assuming.
    metric_crs = base.estimate_utm_crs()
    assert metric_crs is not None
    projected = base.to_crs(metric_crs).iloc[0]
    buffered = projected.buffer(BUFFER_M)
    # Light simplification (≤1 m) controls request size; actual 250 m buffer remains explicit.
    buffered = buffered.simplify(1.0, preserve_topology=True)
    assert buffered.is_valid and not buffered.is_empty
    buffered_wgs = gpd.GeoSeries([buffered], crs=metric_crs).to_crs('EPSG:4326').iloc[0]
    AOI[site] = {'shapely':buffered, 'crs':metric_crs,
                 'ee':ee.Geometry(mapping(buffered_wgs), geodesic=False),
                 'geojson':mapping(buffered_wgs)}
    audit.append({'site':site,'uav_reference_year':YEAR_BY_SITE[site],
                  'source_mosaics':len(intersections[site]),
                  'roi_ha':round(projected.area/1e4,3),
                  'buffered_ha':round(buffered.area/1e4,3),
                  'buffer_m':BUFFER_M,'metric_crs':str(metric_crs),
                  'approx_uint16_fourband_GB_per_month':round(buffered.area/(EXPORT_SCALE_M**2)*8/1e9,2)})

display(pd.DataFrame(audit))
(CACHE_META/'05_site_aoi_audit.csv').write_text(pd.DataFrame(audit).to_csv(index=False))
(CACHE_META/'05_site_aoi_geometries.geojson').write_text(json.dumps({
 'type':'FeatureCollection','features':[{'type':'Feature','properties':{'site':s},'geometry':AOI[s]['geojson']} for s in RUN_SITES]
}))
print('Site-specific ROI + 250 m buffer geometry audited; no imagery exported yet.')

## 03 · Exact one-month selection: do not aggregate or replace missing months

In [ ]:
def month_dates(year, month):
    start = datetime.date(year, month, 1)
    end = datetime.date(year + (month == 12), (month % 12) + 1, 1)
    return start.isoformat(), end.isoformat()

selected = {}
rows = []
for site in RUN_SITES:
    year = YEAR_BY_SITE[site]
    for q, month in QUARTER_MONTHS.items():
        begin, end = month_dates(year, month)
        subset = (COLLECTION.filterDate(begin, end)
                   .filterBounds(AOI[site]['ee'])
                   .filter(ee.Filter.eq('cadence','monthly')))
        number = int(subset.size().getInfo())
        record = {'site':site,'year':year,'quarter':q,'month':month,
                  'requested_month_start':begin, 'requested_month_end_exclusive':end,
                  'matching_monthly_images':number,
                  'status':'available' if number == 1 else ('missing' if number == 0 else 'ambiguous')}
        if number == 1:
            image = ee.Image(subset.first())
            info = ee.Dictionary({
                'id':image.id(),
                'index':image.get('system:index'),
                'time_start':image.get('system:time_start'),
                'time_end':image.get('system:time_end'),
                'cadence':image.get('cadence'),
                'bands':image.bandNames()
            }).getInfo()
            record.update({'image_id':info.get('id'),'image_index':info.get('index'),
                           'time_start_ms':info.get('time_start'), 'time_end_ms':info.get('time_end'),
                           'cadence':info.get('cadence'), 'bands':','.join(info.get('bands',[]))})
            assert set(BANDS).issubset(set(info.get('bands',[])))
            selected[(site,q)] = {'image':image, 'id':info.get('id'), 'year':year, 'month':month}
        rows.append(record)
SELECTION = pd.DataFrame(rows)
display(SELECTION)
SELECTION.to_csv(CACHE_META/'05_planet_monthly_selection.csv',index=False)
assert not (SELECTION.status == 'ambiguous').any(), 'Ambiguous month: inspect catalogue before exporting.'
print('Selected:',len(selected),'of',len(SELECTION),'requested month/site combinations.')
print('Expected Yoko 2020: Q1 and Q2 missing (monthly archive begins September 2020).')
print('No temporal substitution; no mosaics combined.')

## 04 · Export to Drive: georeferenced, lossless source-band input layers

In [ ]:
TASK_REGISTRY = CACHE_META/'05_export_registry_nested_input_layers_v2.json'  # Separate from obsolete root-of-Drive tasks.
if TASK_REGISTRY.exists():
    registry = json.loads(TASK_REGISTRY.read_text())
else:
    registry = {}

def save_registry():
    TASK_REGISTRY.write_text(json.dumps(registry, indent=2, sort_keys=True))

def ee_task_state(task_id):
    if not task_id:
        return 'UNKNOWN'
    try:
        res = ee.data.getTaskStatus(task_id)
        return res[0].get('state','UNKNOWN') if res else 'UNKNOWN'
    except Exception as exc:
        return 'UNKNOWN ('+str(exc)[:80]+')'

# Preserve queued/running/finished task handles even if the Colab runtime restarts.
active_by_description = {}
try:
    for t in ee.batch.Task.list():
        st=t.status()
        if st.get('state') in {'READY','RUNNING','COMPLETED'}:
            active_by_description.setdefault(st.get('description'), []).append(st)
except Exception as exc:
    print('Task-list lookup unavailable:', str(exc)[:200])

new_tasks = 0
for site in RUN_SITES:
    for q, month in QUARTER_MONTHS.items():
        key = f'{site}_{YEAR_BY_SITE[site]}_{q}_M{month:02d}'
        if (site,q) not in selected:
            print('MISSING MONTH (not exported):', key)
            continue
        data = selected[(site,q)]
        image_id = data['id']
        name = f'NB05_Planet_{key}_DN4'
        prefix = name
        task_description = 'MUSANGA_V31_NESTED_' + name
        current = registry.get(key,{})
        if current and current.get('image_id') != image_id:
            raise RuntimeError(f'Image changed for cached key {key}; manual review needed.')
        local_parts = list(CACHE_DRIVE.glob(prefix+'*.tif')) if CACHE_DRIVE.exists() else []
        old_state = ee_task_state(current.get('task_id')) if current else 'NOT_SUBMITTED'
        active = active_by_description.get(task_description, [])
        if SKIP_EXISTING_OR_ACTIVE and (old_state in ('READY','RUNNING','COMPLETED') or active or local_parts):
            print('SKIP existing/active:',key,'old task state:',old_state,
                  'files now visible:',len(local_parts))
            if active and not current.get('task_id'):
                current['task_id'] = active[0].id
                current['description'] = task_description
                current['image_id'] = image_id
                registry[key]=current
            continue
        if not SUBMIT_EXPORTS:
            print('READY FOR EXPORT:',key,'→',name)
            continue
        if new_tasks >= MAX_NEW_TASKS_PER_RUN:
            print('Batch cap reached; rerun this cell when tasks finish.')
            continue
        image = data['image'].select(BANDS).toUint16().clip(AOI[site]['ee'])
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=task_description,
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=prefix,
            region=AOI[site]['ee'],
            crs=EXPORT_CRS,
            scale=EXPORT_SCALE_M,
            fileDimensions=FILE_DIMENSIONS,
            shardSize=256,
            skipEmptyTiles=True,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized':True}
        )
        task.start()
        registry[key] = {
            'site':site, 'quarter':q,'year':data['year'],'month':month,
            'image_id':image_id,'task_id':task.id,'description':task_description,
            'drive_folder':DRIVE_EXPORT_FOLDER, 'prefix':prefix,
            'band_order':BANDS,'pixel_type':'uint16', 'DN_scale_factor':0.0001,
            'crs':EXPORT_CRS, 'nominal_scale_m':EXPORT_SCALE_M,
            'roi_buffer_m':BUFFER_M, 'file_dimensions':FILE_DIMENSIONS,
            'note':'individual monthly NICFI/TFO image; no temporal aggregation'
        }
        save_registry()
        new_tasks += 1
        print('SUBMITTED:',key,'task',task.id)
save_registry()
print('New jobs submitted:',new_tasks,'| registry:',TASK_REGISTRY)

## 05 · Monitor, verify and audit cache; never mistake submitted for downloaded

In [ ]:
summary = []
for key,item in registry.items():
    task_state = ee_task_state(item.get('task_id'))
    prefix=item['prefix']
    files = sorted(CACHE_DRIVE.glob(prefix+'*.tif')) if CACHE_DRIVE.exists() else []
    nbytes=sum(f.stat().st_size for f in files)
    # A large export may be split across many GeoTIFFs; number of parts is variable.
    visible = task_state == 'COMPLETED' and bool(files)
    # File visibility is NOT full radiometric or spatial integrity; Section 06 checks that.
    summary.append({'site_quarter':key,'EE_task_state':task_state,'n_tif_parts':len(files),
                    'total_GB':round(nbytes/1e9,3),'completed_file_visible':visible,
                    'image_id':item['image_id']})
    if task_state == 'FAILED':
        print('FAILED TASK:',key,item.get('task_id'),'— check Earth Engine task details')
AUDIT = pd.DataFrame(summary)
display(AUDIT)
AUDIT.to_csv(CACHE_META/'05_planet_export_status_nested_v2.csv',index=False)
print('Completed + file-visible quarters (run Section 06 for full QA):',int(AUDIT.completed_file_visible.sum()) if len(AUDIT) else 0)
print('Drive folder:',CACHE_DRIVE)
print('CAUTION: exported NICFI monthly mosaics are already multi-scene products;')
print('these cached data contain NO further temporal aggregation by NB05.')

## 06 · Full-scene integrity QA: spatial coverage and radiometry

Scan every GeoTIFF blockwise to distinguish all-zero imagery from zero-filled areas outside an irregular ROI. Report nonzero coverage, positive four-band coverage and values sampled from genuine data. **Do not interpret a top-left zero-valued window as a failed export.**

In [ ]:
# 06 — COMPLETE, BLOCKWISE, FOUR-BAND RASTER QA (never rely on corner pixels)
# Read every raster in blocks; no full-scene memory allocation.
import numpy as np, pandas as pd, rasterio
from pathlib import Path
from IPython.display import display

paths = sorted(CACHE_DRIVE.glob('NB05_Planet_*.tif'))
assert CACHE_DRIVE.is_dir(), CACHE_DRIVE
if not paths:
    print('No GeoTIFF parts visible in expected nested folder:', CACHE_DRIVE)
else:
    qa_rows = []
    for path in paths:
        with rasterio.open(path) as ds:
            assert ds.count == 4 and all(d == 'uint16' for d in ds.dtypes), (path,ds.count,ds.dtypes)
            assert ds.crs is not None, path
            total = ds.width * ds.height
            valid_all = 0
            any_nonzero = 0
            max_band = np.zeros(4, dtype=np.uint16)
            min_positive_band = [None]*4
            good_samples = []
            for _, window in ds.block_windows(1):
                # GEE exports often store off-ROI pixels as zero with no NoData metadata.
                a = ds.read(window=window, masked=False)
                any_valid = np.any(a > 0, axis=0)
                all_valid = np.all(a > 0, axis=0)
                any_nonzero += int(any_valid.sum())
                valid_all += int(all_valid.sum())
                max_band = np.maximum(max_band, a.reshape(4,-1).max(axis=1))
                for b in range(4):
                    positive = a[b][a[b]>0]
                    if positive.size:
                        lo = int(positive.min())
                        min_positive_band[b] = lo if min_positive_band[b] is None else min(lo,min_positive_band[b])
                if len(good_samples)<4 and all_valid.any():
                    rr,cc = np.argwhere(all_valid)[len(np.argwhere(all_valid))//2]
                    good_samples.append((int(window.row_off+rr), int(window.col_off+cc), a[:,rr,cc].tolist()))
            status = 'OK' if any_nonzero>0 and valid_all>0 else 'EMPTY_OR_ZERO'
            qa_rows.append(dict(file=path.name,status=status,crs=str(ds.crs),
                width=ds.width,height=ds.height,mb=round(path.stat().st_size/1e6,2),
                any_nonzero_pct=round(100*any_nonzero/total,2),
                all_four_positive_pct=round(100*valid_all/total,2),
                band_max_B_G_R_N=max_band.tolist(),
                band_min_positive_B_G_R_N=min_positive_band,
                example_valid_pixel=good_samples[0] if good_samples else None))
    qa = pd.DataFrame(qa_rows)
    display(qa)
    qa.to_csv(CACHE_META/'05_planet_full_raster_quality.csv',index=False)
    print('Important: zero is being treated as exported background, not guaranteed NoData.')
    print('If zero-only: check geometry, exported region, EE task details and mask handling.')
    print('If complete EE task has no visible file: check Drive indexing/location, do NOT resubmit blindly.')


## Scientific safeguards and next experiments

- The new full-year pilot is **Baego 2023 only**. No MOS/Yangambi/Yoko exports are submitted by default.
- Existing Baego March, June, September and December files are retained, **not deleted or re-exported**.
- The annual median uses a documented **heuristic temporal brightness-anomaly filter** (brightness increase >0.08 reflectance units together with NDVI decrease >0.10). It does not provide an official cloud mask. Compare with unscreened median and raw/screened counts.
- Annual texture is a temporal median of **per-month normalized-NIR texture**, not texture of the annual reflectance median.
- No temporal substitution, no downstream model fitting, no change to NB04's validated satellite predictors.
- Complete the Baego QA before expanding to other sites. Maintain identical folds and target grid for subsequent predictor ablation.


## 07 · Baego 2023 — complete 12-month NICFI inventory and safe reuse

Run Sections 00–02 first. This section identifies **all 12 individual monthly NICFI mosaics**, checks prior Q1–Q4 files, and lists only genuinely missing files for export. **Do not use quarterly selection from Section 03 as the full-year inventory.**


In [ ]:
# 07 — Catalogue all 12 Baego monthly mosaics; no exports.
PILOT_SITE = 'Baego'
PILOT_YEAR = 2023
assert PILOT_SITE in AOI
from collections import OrderedDict
BAEGO_MONTH_IMAGES = {}
BAEGO_MONTH_RECORDS = []
BAEGO_MONTHLY_FILES = {}
for month in range(1,13):
    start,end = month_dates(PILOT_YEAR, month)
    subset = (COLLECTION.filterBounds(AOI[PILOT_SITE]['ee'])
              .filterDate(start,end)
              .filter(ee.Filter.eq('cadence','monthly')))
    count = int(subset.size().getInfo())
    if count != 1:
        BAEGO_MONTH_RECORDS.append(dict(month=month,status='MISSING' if count==0 else 'AMBIGUOUS',
                                        image_id='',local_file='',n_matches=count))
        continue
    img = ee.Image(subset.first())
    imgid = img.id().getInfo()
    # Preserve four already downloaded GeoTIFFs; do not rename or duplicate.
    if month in (3,6,9,12):
        q = {3:1,6:2,9:3,12:4}[month]
        prefix = f'NB05_Planet_Baego_2023_Q{q}_M{month:02d}_DN4'
    else:
        prefix = f'NB05_Planet_Baego_2023_M{month:02d}_DN4'
    files = sorted(CACHE_DRIVE.glob(prefix+'*.tif'))
    BAEGO_MONTH_IMAGES[month] = img
    BAEGO_MONTHLY_FILES[month] = prefix
    BAEGO_MONTH_RECORDS.append(dict(month=month,status='CACHED' if files else 'TO_EXPORT',
                                    image_id=imgid,local_file=prefix,n_matches=count,
                                    parts_visible=len(files)))
BAEGO_CATALOGUE = pd.DataFrame(BAEGO_MONTH_RECORDS)
display(BAEGO_CATALOGUE)
BAEGO_CATALOGUE.to_csv(CACHE_META/'07_baego_full_year_catalogue.csv',index=False)
assert len(BAEGO_MONTH_IMAGES)==12, 'STOP: 2023 Baego does not have 12 unambiguous monthly mosaics.'
print('Months visible:', BAEGO_CATALOGUE.query("status=='CACHED'").month.tolist())
print('Months missing locally:', BAEGO_CATALOGUE.query("status=='TO_EXPORT'").month.tolist())


## 08 · Export eight missing Baego months to the existing nested Drive folder

Turn `SUBMIT_MISSING_BAEGO_MONTHS=True` in Section 00 and rerun the settings cell plus this cell. Maximum four new jobs per run. Existing four quarterly files are retained. Task IDs use a **new dedicated full-year registry**, and the operation never deletes data. Earth Engine tasks continue after Colab disconnects. Rerun Section 08 to submit the remaining months.


In [ ]:
# 08 — Additional 2023 monthly GeoTIFFs (new jobs only).
FULLYEAR_REGISTRY_PATH = CACHE_META/'08_baego_full_year_monthly_tasks.json'
FY_REG = json.loads(FULLYEAR_REGISTRY_PATH.read_text()) if FULLYEAR_REGISTRY_PATH.exists() else {}
def save_fy_registry():
    FULLYEAR_REGISTRY_PATH.write_text(json.dumps(FY_REG,indent=2,sort_keys=True))
active_tasks = {}
for task in ee.batch.Task.list():
    st=task.status()
    if st.get('state') in ('READY','RUNNING','COMPLETED'):
        active_tasks.setdefault(st.get('description'), []).append(task)
submitted = 0
for month in range(1,13):
    prefix=BAEGO_MONTHLY_FILES[month]
    name=f'MUSANGA_V31_BAEGO2023_FULLYEAR_{month:02d}'
    files=sorted(CACHE_DRIVE.glob(prefix+'*.tif'))
    if files:
        print(f'{month:02d}: CACHED ({len(files)} part(s)); not resubmitted.')
        continue
    if month in (3,6,9,12):
        print(f'{month:02d}: Existing quarterly export missing; inspect Section 04 registry; not recreated here.')
        continue
    prior=FY_REG.get(str(month),{})
    state=ee_task_state(prior.get('task_id')) if prior else 'UNKNOWN'
    if name in active_tasks or state in ('READY','RUNNING','COMPLETED'):
        print(f'{month:02d}: existing EE task ({state}); skip.')
        continue
    if not SUBMIT_MISSING_BAEGO_MONTHS:
        print(f'{month:02d}: READY; export switch is False.')
        continue
    if submitted >= MAX_NEW_TASKS_PER_RUN:
        print(f'{month:02d}: deferred; batch cap {MAX_NEW_TASKS_PER_RUN}.')
        continue
    image = BAEGO_MONTH_IMAGES[month].select(BANDS).toUint16().clip(AOI['Baego']['ee'])
    task = ee.batch.Export.image.toDrive(
        image=image, description=name, folder=DRIVE_EXPORT_FOLDER,
        fileNamePrefix=prefix, region=AOI['Baego']['ee'],
        crs=EXPORT_CRS,scale=EXPORT_SCALE_M,
        fileDimensions=FILE_DIMENSIONS,shardSize=256,
        skipEmptyTiles=True,maxPixels=1e13,fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized':True})
    task.start()
    FY_REG[str(month)]=dict(task_id=task.id,image_id=BAEGO_MONTH_IMAGES[month].id().getInfo(),
        description=name,prefix=prefix,folder=DRIVE_EXPORT_FOLDER)
    save_fy_registry()
    submitted+=1
    print(f'{month:02d}: SUBMITTED {task.id}')
print('New monthly jobs:', submitted)
print('Registry:',FULLYEAR_REGISTRY_PATH)


## 09 · Annual cloud-screened reflectance (Baego pilot)

Use the **12 monthly mosaics directly in Earth Engine**, without waiting for all local exports. Export: 4-band annual median, a month count and a conservative temporal brightness-outlier count, plus an unscreened annual median as a control. Pixels are masked if their mean visible reflectance rises unusually above their own 12-month baseline **and** NDVI falls. This is *heuristic screening*, not official cloud masking. The output is cached in a dedicated `input_layers` subfolder. Inspect both screened and unscreened annual images.


In [ ]:
# 09 — Annual 4-band median and conservative cloud-like temporal anomalies.
ANNUAL_DRIVE_FOLDER = 'Musanga_V3_1_05_Planet_Annual'
ANNUAL_CACHE = INPUT_ROOT/ANNUAL_DRIVE_FOLDER
ANNUAL_CACHE.mkdir(parents=True,exist_ok=True) # precreate unique nested Drive folder
from functools import reduce
raw_images=[BAEGO_MONTH_IMAGES[m].select(BANDS).toFloat() for m in range(1,13)]
def valid_ee(img):
    return img.updateMask(img.select(BANDS).reduce(ee.Reducer.min()).gt(0))
valid_collection=ee.ImageCollection.fromImages([valid_ee(im) for im in raw_images])
baseline=valid_collection.median()
def mask_bright_outlier(img):
    v=img.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
    base=baseline.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
    ndvi=img.normalizedDifference(['N','R'])
    base_ndvi=baseline.normalizedDifference(['N','R'])
    # Bright visible anomaly >0.08 reflectance units AND NDVI falls >0.10.
    # A conservative temporal screening heuristic; NOT validated cloud labels.
    cloudlike=v.subtract(base).gt(0.08).And(base_ndvi.subtract(ndvi).gt(0.10))
    return img.updateMask(cloudlike.Not()).set('cloudscreen','conservative_temporal_brightness')
PLANET_BAEGO_SCREENED_IC = valid_collection.map(mask_bright_outlier)  # Persistent EE collection: never reuse 'screened' for NumPy arrays
unscreened_median=valid_collection.median().rename(['B','G','R','N'])
screened_median=PLANET_BAEGO_SCREENED_IC.median().rename(['B','G','R','N'])
count_valid=valid_collection.select('N').count().rename('months_raw_valid')
count_screened=PLANET_BAEGO_SCREENED_IC.select('N').count().rename('months_screened_valid')
removed=count_valid.subtract(count_screened).rename('months_screened_out')
# Original DN bands are 0..10000. Median products are float and preserved in same DN units.
annual_image=ee.Image.cat([screened_median,unscreened_median.rename(['B_raw','G_raw','R_raw','N_raw']),
                           count_valid,count_screened,removed]).toFloat().clip(AOI['Baego']['ee'])
ANNUAL_PREFIX='NB05_Baego_2023_annual_median_screened_vs_raw_11band'
annual_path=ANNUAL_CACHE/ (ANNUAL_PREFIX+'.tif')
annual_desc='MUSANGA_V31_BAEGO2023_ANNUAL_REFLECTANCE'
annual_task_registry=CACHE_META/'09_baego_annual_export.json'
annual_record=json.loads(annual_task_registry.read_text()) if annual_task_registry.exists() else {}
print('Annual bands:',annual_image.bandNames().getInfo())
print('Export destination:',annual_path)
if annual_path.exists():
    print('SKIP: annual GeoTIFF already visible. Inspect quality before trusting contents.')
elif annual_record.get('task_id') and ee_task_state(annual_record.get('task_id')) in ('READY','RUNNING','COMPLETED'):
    print('SKIP: annual export exists:',ee_task_state(annual_record.get('task_id')))
elif SUBMIT_ANNUAL_PRODUCTS:
    task=ee.batch.Export.image.toDrive(
        image=annual_image,description=annual_desc,folder=ANNUAL_DRIVE_FOLDER,
        fileNamePrefix=ANNUAL_PREFIX,region=AOI['Baego']['ee'],
        crs=EXPORT_CRS,scale=EXPORT_SCALE_M,
        fileDimensions=FILE_DIMENSIONS,shardSize=256,
        skipEmptyTiles=True,maxPixels=1e13,fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized':True})
    task.start()
    annual_record={'task_id':task.id,'prefix':ANNUAL_PREFIX,
        'folder':ANNUAL_DRIVE_FOLDER,'bands':annual_image.bandNames().getInfo()}
    annual_task_registry.write_text(json.dumps(annual_record,indent=2))
    print('SUBMITTED annual reflectance:',task.id)
else:
    print('READY — set SUBMIT_ANNUAL_PRODUCTS=True in cell 00 to export.')


### 09B · Preview screened vs unscreened annual reflectance before export

Interactive Earth Engine visualization — no data downloaded. The reflectance contrast is fixed across both layers. Show the masked-observation count to locate suspected clouds and residual haze. The mask is an experiment, not an official Planet quality layer.


In [ ]:
# 09B — Interactive Baego annual preview (no exports)
import geemap
preview = geemap.Map()
preview.centerObject(AOI['Baego']['ee'], 13)
rgb_vis = dict(bands=['R','G','B'],min=300,max=3500,gamma=1.3)
preview.addLayer(unscreened_median.clip(AOI['Baego']['ee']),rgb_vis,
                 'Annual median (no custom screening)',True)
preview.addLayer(screened_median.clip(AOI['Baego']['ee']),rgb_vis,
                 'Annual median (screened)',False)
preview.addLayer(count_screened.clip(AOI['Baego']['ee']),
                 dict(min=0,max=12,palette=['red','orange','yellow','lightgreen','darkgreen']),
                 'Screened valid-month count',False)
preview.addLayer(removed.clip(AOI['Baego']['ee']),
                 dict(min=0,max=5,palette=['black','yellow','orange','red']),
                 'Months screened out (heuristic)',False)
display(preview)


## 10 · Annual texture from monthly normalized NIR, not an annual-mean image

For **each cloud-screened month**, compute local NIR coefficient of variation at three spatial scales (~14, 33 and 71 m), plus a normalized absolute high-pass signal (~24 m). Then derive annual *medians* and temporal standard deviations of the four monthly-texture bands. Also save a valid-month count. This approach retains spatial texture without requiring pixelwise perfect annual-image registration. These are exploratory features; test stability and sensitivity to image sharpness before using them for Musanga prediction.


In [ ]:
# 10 — Native-resolution texture operators per month, then annual temporal summaries.
# Non-learned texture baseline; CNN and LBP can follow after this has been validated.
TEXTURE_DRIVE_FOLDER='Musanga_V3_1_05_Planet_Texture'
TEXTURE_CACHE=CACHE_META/'derived_features'/TEXTURE_DRIVE_FOLDER
TEXTURE_CACHE.mkdir(parents=True,exist_ok=True)
def monthly_texture(img):
    nir=img.select('N').multiply(0.0001)
    layers=[]
    for radius in (1,3,7): # radius pixels at ~4.77m resolution
        kernel=ee.Kernel.square(radius=radius,units='pixels',normalize=False)
        avg=nir.reduceNeighborhood(reducer=ee.Reducer.mean(),kernel=kernel)
        sd=nir.reduceNeighborhood(reducer=ee.Reducer.stdDev(),kernel=kernel)
        cv=sd.divide(avg.abs().max(ee.Image.constant(0.02))).rename(f'NIR_CV_r{radius}')
        layers.append(cv)
    mean5=nir.focal_mean(radius=2,units='pixels')
    highpass=nir.subtract(mean5).abs().divide(mean5.abs().max(ee.Image.constant(0.02)))
    layers.append(highpass.rename('NIR_abs_highpass_r2'))
    return ee.Image.cat(layers).toFloat().copyProperties(img,['system:time_start'])
# Rebuild the collection from the original Planet inputs to avoid notebook variable clashes.
# In particular, plotting cells may assign a NumPy array to a variable named 'screened'.
assert isinstance(valid_collection, ee.imagecollection.ImageCollection), 'Run Section 09 first.'
PLANET_BAEGO_TEXTURE_SOURCE_IC = valid_collection.map(mask_bright_outlier)
texture_collection=PLANET_BAEGO_TEXTURE_SOURCE_IC.map(monthly_texture)
texture_median=texture_collection.median()
texture_sd=texture_collection.reduce(ee.Reducer.stdDev())
texture_sd=texture_sd.rename([v+'_temporal_sd' for v in texture_median.bandNames().getInfo()])
texture_count=texture_collection.select('NIR_CV_r3').count().rename('texture_valid_month_count')
texture_image=ee.Image.cat([texture_median,texture_sd,texture_count]).toFloat().clip(AOI['Baego']['ee'])
TEXTURE_PREFIX='NB05_Baego_2023_annual_monthlyNIR_texture_9band'
TEXTURE_PATH=TEXTURE_CACHE/(TEXTURE_PREFIX+'.tif')
TEXTURE_REG_PATH=CACHE_META/'10_baego_texture_export.json'
text_record=json.loads(TEXTURE_REG_PATH.read_text()) if TEXTURE_REG_PATH.exists() else {}
print('Texture export bands:',texture_image.bandNames().getInfo())
print('Destination:',TEXTURE_PATH)
if TEXTURE_PATH.exists():
    print('SKIP existing texture output. Inspect first.')
elif text_record.get('task_id') and ee_task_state(text_record.get('task_id')) in ('READY','RUNNING','COMPLETED'):
    print('SKIP existing EE task:',ee_task_state(text_record.get('task_id')))
elif SUBMIT_ANNUAL_PRODUCTS:
    task=ee.batch.Export.image.toDrive(
        image=texture_image,description='MUSANGA_V31_BAEGO2023_ANNUAL_TEXTURE',
        folder=TEXTURE_DRIVE_FOLDER,fileNamePrefix=TEXTURE_PREFIX,
        region=AOI['Baego']['ee'],crs=EXPORT_CRS,scale=EXPORT_SCALE_M,
        fileDimensions=FILE_DIMENSIONS,shardSize=256,
        skipEmptyTiles=True,maxPixels=1e13,fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized':True})
    task.start()
    TEXTURE_REG_PATH.write_text(json.dumps(dict(task_id=task.id,prefix=TEXTURE_PREFIX,
        folder=TEXTURE_DRIVE_FOLDER,bands=texture_image.bandNames().getInfo()),indent=2))
    print('SUBMITTED texture:',task.id)
else:
    print('READY — set SUBMIT_ANNUAL_PRODUCTS=True in cell 00 to export.')


## 11 · Inspect exported annual GeoTIFFs and compare to available monthly input layers

Run this after the Earth Engine jobs finish. This is an integrity check: report file sizes, band counts, georeferencing and nonzero pixels. For the cloud-mask diagnostic compare annual `months_raw_valid` vs `months_screened_valid`, and inspect RGB imagery. Do not infer scientific cloud-mask accuracy from the heuristic alone.


In [ ]:
# 11 — Files, task statuses and blockwise image support audit (no model fitting).
for label,folder,prefix,regpath in [
    ('reflectance',ANNUAL_CACHE,ANNUAL_PREFIX,annual_task_registry),
    ('texture',TEXTURE_CACHE,TEXTURE_PREFIX,TEXTURE_REG_PATH)]:
    paths=sorted(folder.glob(prefix+'*.tif'))
    record=json.loads(regpath.read_text()) if regpath.exists() else {}
    print('\n',label.upper(),'EE state:',ee_task_state(record.get('task_id')),
          'GeoTIFF parts:',len(paths),'folder:',folder)
    for path in paths:
        with rasterio.open(path) as ds:
            # Read window-by-window; avoid materializing giant mosaics in RAM.
            nonzero=0
            for _,w in ds.block_windows(1):
                nonzero+=int(np.count_nonzero(ds.read(1,window=w)))
            print(path.name,'bands',ds.count,'shape',(ds.height,ds.width),
                  'crs',ds.crs,'nonzero band-1 px',nonzero,
                  'size_MB',round(path.stat().st_size/1e6,2))
print('NOTE: Earth Engine can complete an export before the mounted Drive folder refreshes.')
